In [11]:
#https://www.cs.toronto.edu/~kriz/cifar.html

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Strong Data Augmentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),        # Random cropping with padding -> robust to shifts
    transforms.RandomHorizontalFlip(),           # Randomly flip image -> invariance to orientation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Color distortion
    transforms.RandomRotation(15),               # Randomly rotate image up to 15 degrees
    transforms.ToTensor(),                       # Convert to tensor
    transforms.Normalize((0.5,), (0.5,))         # Normalize pixel values into [-1, 1]
])


test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])  #Test set should not use heavy augmentation -> only normalization.

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=False, transform=train_transform)  #CIFAR-10 training data (50k images, 32x32, 10 classes).
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=False, transform=test_transform)   #CIFAR-10 test data (10k images).

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)    #Loads batches of 128 samples for training, with shuffling.
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)    #Loads test data in batches of 128, no shuffle needed.

# Deep CNN - VGG-like 8 layer
class DeepCNN(nn.Module):
    def __init__(self):
        super(DeepCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),   #First feature block: two conv layers (3x3 kernels, padding=1 keeps size same).
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  #Output channels = 64.
            nn.MaxPool2d(2, 2),  # Then max-pool reduces spatial size from 32x32 -> 16x16.

            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # Second block: expands to 128 channels, then pooling reduces size to 8x8.

            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # Third block: expands to 256 channels, then pooling reduces size to 4x4.
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                            # Flatten 256x4x4 -> 4096
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),  # Fully connected hidden layer (4096 -> 512)
            nn.Dropout(0.3),                         # Dropout to prevent overfitting
            nn.Linear(512, 10)                       # Final classification layer (512 -> 10 classes)
        )


    def forward(self, x):
        x = self.features(x)     # Extract features with CNN blocks
        x = self.classifier(x)   # Classify using dense layers
        return x


model = DeepCNN().to(device)
print(model)

# Label smoothing loss
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=10, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.criterion = nn.KLDivLoss(reduction='batchmean')
    
    # Implements Label Smoothing -> avoids overconfidence by softening one-hot labels.
    # Example: instead of [0,0,1,0,...], it becomes [0.01, 0.01, 0.91, 0.01,...].

    def forward(self, x, target):
        true_dist = torch.zeros_like(x)   # Initialize all probs to zero
        true_dist.fill_(self.smoothing / (self.cls - 1))  # Fill all other classes with small smoothing value
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)  # Set correct class prob = 1 - smoothing
        return self.criterion(torch.log_softmax(x, dim=1), true_dist)    #Uses KL divergence between smoothed target distribution and predicted softmax distribution.


criterion = LabelSmoothingLoss(classes=10, smoothing=0.1)

# Optimizer + Cosine LR Scheduler
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# Training
num_epochs = 50
for epoch in range(num_epochs):
    model.train()                # Training mode (enables dropout + BN updates)
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)         # Forward pass
        loss = criterion(outputs, labels)  # Compute smoothed loss

        optimizer.zero_grad()   # Reset gradients
        loss.backward()         # Backpropagation
        optimizer.step()        # Update weights

        total_loss += loss.item()


    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

# Testing
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Test Accuracy: {acc:.2f}%")


Running on: cuda
DeepCNN(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 256, kernel_size=(3, 3), strid

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on: {device}")

# Improved Data Augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.2)
])


transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# Stronger CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN().to(device)

# CrossEntropy with Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=70)

# ADMM params
rho = 1e-4
admm_Z, admm_U = {}, {}
target_layers = [name for name, _ in model.named_parameters() if 'weight' in name]

for name, param in model.named_parameters():
    if name in target_layers:
        admm_Z[name] = param.detach().clone()
        admm_U[name] = torch.zeros_like(param).to(device)

# Warm-up training (10 epochs)
print("Warm-up phase...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Warm-up Epoch [{epoch+1}/10] Loss: {total_loss/len(train_loader):.4f}")

# ADMM pruning phase (30 epochs)
print("ADMM pruning phase...")
for epoch in range(30):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        admm_penalty = 0
        for name, param in model.named_parameters():
            if name in target_layers:
                z = admm_Z[name]
                u = admm_U[name]
                admm_penalty += (rho / 2) * torch.norm(param - z + u) ** 2

        total = loss + admm_penalty

        optimizer.zero_grad()
        total.backward()
        optimizer.step()
        total_loss += total.item()

    # Update Z, U
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in target_layers:
                temp = param + admm_U[name]
                threshold = rho
                admm_Z[name] = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
                admm_U[name] = admm_U[name] + (param - admm_Z[name])

    scheduler.step()
    print(f"ADMM Epoch [{epoch+1}/30] Loss: {total_loss/len(train_loader):.4f}")

# Fine-tuning (10 epochs)
print("Fine-tuning pruned model...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Fine-tune Epoch [{epoch+1}/10] Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Final Accuracy after ADMM pruning + fine-tuning: {acc:.2f}%")

torch.save(model.state_dict(), 'cnn_admm_pruned_improved.pth')


Running on: cuda
Warm-up phase...
Warm-up Epoch [1/10] Loss: 1.8908
Warm-up Epoch [2/10] Loss: 1.5061
Warm-up Epoch [3/10] Loss: 1.3521
Warm-up Epoch [4/10] Loss: 1.2538
Warm-up Epoch [5/10] Loss: 1.1807
Warm-up Epoch [6/10] Loss: 1.1209
Warm-up Epoch [7/10] Loss: 1.0726
Warm-up Epoch [8/10] Loss: 1.0348
Warm-up Epoch [9/10] Loss: 1.0024
Warm-up Epoch [10/10] Loss: 0.9730
ADMM pruning phase...
ADMM Epoch [1/30] Loss: 1.1233
ADMM Epoch [2/30] Loss: 0.9448
ADMM Epoch [3/30] Loss: 0.9208
ADMM Epoch [4/30] Loss: 0.8940
ADMM Epoch [5/30] Loss: 0.8760
ADMM Epoch [6/30] Loss: 0.8623
ADMM Epoch [7/30] Loss: 0.8474
ADMM Epoch [8/30] Loss: 0.8291
ADMM Epoch [9/30] Loss: 0.8183
ADMM Epoch [10/30] Loss: 0.8030
ADMM Epoch [11/30] Loss: 0.7941
ADMM Epoch [12/30] Loss: 0.7803
ADMM Epoch [13/30] Loss: 0.7694
ADMM Epoch [14/30] Loss: 0.7624
ADMM Epoch [15/30] Loss: 0.7504
ADMM Epoch [16/30] Loss: 0.7449
ADMM Epoch [17/30] Loss: 0.7312
ADMM Epoch [18/30] Loss: 0.7279
ADMM Epoch [19/30] Loss: 0.7193
ADMM